# Person/Actor Architecture - User, Client, and Collaborator Management

This notebook demonstrates the new Person/Actor architecture in the Siege Utilities library, which supports:

- **Users**: Siege/Masai team members with credentials and preferences
- **Clients**: External client organizations with branding and project data
- **Collaborators**: External collaborators (like Tony from Masai) with access controls
- **Organizations**: Companies (Siege, Masai, Hillcrest) with member management
- **Collaborations**: Joint projects between organizations

## Key Features:
- **Credential Management**: API keys, OAuth tokens, database connections
- **Relationship Management**: Users assigned to clients, collaborators in projects
- **Import/Export**: Full backup and sharing capabilities
- **Security**: Sensitive data can be excluded from exports

## Hillcrest Information (from PDF):
- **Full Name**: Hillcrest Children & Family Center
- **Industry**: Nonprofit - Children & Family Services  
- **Location**: Washington, DC area
- **Focus**: Marketing analytics for donor engagement and program visibility

In [10]:
# Import Person/Actor architecture modules
import sys
from pathlib import Path
import siege_utilities as su

# Initialize logging
su.configure_shared_logging(level="INFO")

# Add parent directory to path for notebook imports
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    sys.path.insert(0, str(notebook_dir.parent))

# Import Person-based models
from siege_utilities.config import (
    Person, User, Client, Collaborator, Organization, Collaboration,
    Credential, OnePasswordCredential, OAuthIntegration, OAuthScope, DatabaseConnection
)

# Note: We'll use the Person/Actor models directly instead of the enhanced_config functions
# which still use the old UserProfile/ClientProfile models

su.log_info("Person/Actor architecture modules imported successfully")

[siege_utilities] 2026-02-03 14:06:39,462 INFO: Person/Actor architecture modules imported successfully
[siege_utilities] 2026-02-03 14:06:39,462 INFO: Person/Actor architecture modules imported successfully
[siege_utilities] 2026-02-03 14:06:39,462 INFO: Person/Actor architecture modules imported successfully
[siege_utilities] 2026-02-03 14:06:39,462 INFO: Person/Actor architecture modules imported successfully
[siege_utilities] 2026-02-03 14:06:39,462 INFO: Person/Actor architecture modules imported successfully


In [ ]:
# 1. Create Organizations
#
# Organization Field Constraints (Pydantic validated):
#   org_id:        1-50 chars, pattern: [a-zA-Z0-9_-]+
#   name:          1-100 chars
#   org_type:      vendor | client | partner | internal
#   primary_email: must match ^[^@]+@[^@]+\.[^@]+$
#   phone:         optional, 10-15 digits (extracted from string)
#   website:       optional, must match ^https?://...
#   address:       optional, max 200 chars
#   status:        active | inactive | archived (default: active)

# Siege Analytics organization
siege_org = Organization(
    org_id='siege_analytics',              # 1-50 chars, [a-zA-Z0-9_-]+
    name='Siege Analytics',                # 1-100 chars
    org_type='internal',                   # vendor | client | partner | internal
    primary_email='dheeraj@siegeanalytics.com',  # valid email format
    phone='+1-512-850-5473',               # optional, 10-15 digits
    website='https://siegeanalytics.com',  # optional, https?:// URL
    address='Austin, TX'                   # optional, max 200 chars
)

# Masai Interactive organization  
masai_org = Organization(
    org_id='masai_interactive',            # 1-50 chars, [a-zA-Z0-9_-]+
    name='Masai Interactive',              # 1-100 chars
    org_type='partner',                    # vendor | client | partner | internal
    primary_email='info@masaiinteractive.com',
    phone=None,
    website='https://masaiinteractive.com',
    address='Washington, DC'
)

# Hillcrest organization
hillcrest_org = Organization(
    org_id='hillcrest_children',           # 1-50 chars, [a-zA-Z0-9_-]+
    name='Hillcrest Children & Family Center',
    org_type='client',                     # vendor | client | partner | internal
    primary_email='info@hillcrestchildren.org',
    phone='+1-202-555-0123',
    website='https://www.hillcrestchildren.org',
    address='Washington, DC'
)

su.log_info("Created organizations:")
su.log_info(f"   - {siege_org.name} ({siege_org.org_type})")
su.log_info(f"   - {masai_org.name} ({masai_org.org_type})")
su.log_info(f"   - {hillcrest_org.name} ({hillcrest_org.org_type})")

In [ ]:
# 2. Create Users with Credentials
#
# Credential Field Constraints (Pydantic validated):
#   name:            1-100 chars, pattern: [a-zA-Z0-9_-]+
#   credential_type: api_key | oauth_token | username_password | ssh_key | certificate | secret
#   service:         1-50 chars, pattern: [a-zA-Z0-9_-]+
#   api_key:         min 10 chars (if provided)
#   password:        min 8 chars (if provided)
#   status:          active | expired | revoked | pending (default: active)
#
# DatabaseConnection Field Constraints (Pydantic validated):
#   name:            1-50 chars, pattern: [a-zA-Z0-9_-]+
#   connection_type: postgresql | mysql | sqlite | duckdb
#   host:            1-255 chars, pattern: [a-zA-Z0-9.-]+
#   port:            1-65535
#   database:        1-50 chars, must start with letter, pattern: [a-zA-Z][a-zA-Z0-9_-]*
#   username:        1-50 chars
#   password:        8-100 chars, REQUIRES: uppercase + lowercase + number
#
# User Field Constraints (extends Person):
#   username:        1-50 chars, pattern: [a-zA-Z0-9_-]+
#   github_login:    optional, max 50 chars, pattern: [a-zA-Z0-9_-]+
#   role:            analyst | manager | developer | admin | collaborator
#
# Collaborator Field Constraints (extends Person):
#   external_organization: 1-100 chars (cannot be empty)
#   collaboration_level:   read | write | admin (default: read)
#   access_expires:        must be in the future (if provided)

from datetime import datetime

# Dheeraj (Siege User) with credentials
dheeraj_ga_cred = Credential(
    name='dheeraj_google_analytics',       # 1-100 chars, [a-zA-Z0-9_-]+
    credential_type='api_key',             # api_key | oauth_token | username_password | ...
    service='google_analytics',            # 1-50 chars, [a-zA-Z0-9_-]+
    api_key='GA-123456789-DHEERAJ',        # min 10 chars
    username='dheeraj@siegeanalytics.com'
)

dheeraj_db_conn = DatabaseConnection(
    name='dheeraj_localpostgis',           # 1-50 chars, [a-zA-Z0-9_-]+
    connection_type='postgresql',          # postgresql | mysql | sqlite | duckdb
    host='localhost',                      # 1-255 chars, [a-zA-Z0-9.-]+
    port=5432,                             # 1-65535
    database='postgres',                   # 1-50 chars, starts with letter
    username='dheeraj',                    # 1-50 chars
    password='Dessert123!'                 # 8-100 chars, uppercase + lowercase + number
)

dheeraj = User(
    person_id='dheeraj_siege',             # 1-50 chars, [a-zA-Z0-9_-]+
    name='Dheeraj Chand',                  # 1-100 chars
    email='dheeraj@siegeanalytics.com',    # valid email format
    username='dheerajchand',               # 1-50 chars, [a-zA-Z0-9_-]+
    github_login='dheerajchand',           # optional, [a-zA-Z0-9_-]+
    role='admin',                          # analyst | manager | developer | admin | collaborator
    organizations=['siege_analytics'],
    primary_organization='siege_analytics',
    credentials=[dheeraj_ga_cred],
    database_connections=[dheeraj_db_conn]
)

# Tony (Masai Collaborator) with limited access
tony_oauth = OAuthIntegration(
    name='google_analytics_oauth',         # 1-100 chars, [a-zA-Z0-9_-]+
    provider='google',                     # google | microsoft | github | linkedin | ...
    service='google_analytics',            # 1-50 chars, [a-zA-Z0-9_-]+
    client_id='tony_google_client_123456789',    # 10-200 chars
    client_secret='tony_secret_123456789',       # 10-200 chars
    redirect_uri='https://masaiinteractive.com/oauth',  # must match https?://...
    scopes=[OAuthScope.ANALYTICS]
)

tony = Collaborator(
    person_id='tony_masai',                # 1-50 chars, [a-zA-Z0-9_-]+
    name='Tony Teat',                      # 1-100 chars
    email='tony@masaiinteractive.com',     # valid email format
    external_organization='Masai Interactive',  # 1-100 chars (required)
    collaboration_level='write',           # read | write | admin
    organizations=['masai_interactive'],
    oauth_integrations=[tony_oauth],
    allowed_services=['google_analytics', 'facebook_insights'],
    access_expires=datetime(2025, 12, 31)  # must be in the future
)

su.log_info("Created users:")
su.log_info(f"   - {dheeraj.name} (User) - {dheeraj.role}")
su.log_info(f"   - {tony.name} (Collaborator) - {tony.collaboration_level}")

In [ ]:
# 3. Create Client with Branding
#
# Client Field Constraints (extends Person):
#   client_code:    2-10 chars, pattern: [A-Z0-9]+ (must be uppercase)
#                   Reserved codes: DEFAULT, ADMIN, SYSTEM, TEST
#   industry:       1-50 chars
#   project_count:  >= 0
#   client_status:  active | inactive | archived
#   branding_config:    optional Dict
#   report_preferences: optional Dict

# Hillcrest client with comprehensive branding
hillcrest_client = Client(
    person_id='hillcrest_client',          # 1-50 chars, [a-zA-Z0-9_-]+
    name='Hillcrest Children & Family Center',  # 1-100 chars
    email='info@hillcrestchildren.org',    # valid email format
    client_code='HILL',                    # 2-10 chars, [A-Z0-9]+, uppercase only
    industry='Nonprofit - Children & Family Services',  # 1-50 chars
    phone='+1-202-555-0123',               # optional, 10-15 digits
    address='Washington, DC',              # optional, max 200 chars
    website='https://www.hillcrestchildren.org',  # optional, https?://...
    linkedin='https://linkedin.com/company/hillcrest-children-family-center',  # linkedin URL pattern
    project_count=1,                       # >= 0
    client_status='active',                # active | inactive | archived
    organizations=['hillcrest_children'],
    primary_organization='hillcrest_children',
    branding_config={
        'primary_color': '#2E8B57',        # Sea Green
        'secondary_color': '#4169E1',      # Royal Blue  
        'accent_color': '#FFD700',         # Gold
        'logo_path': 'hillcrest_logo.png'
    },
    report_preferences={
        'default_format': 'pdf',
        'chart_style': 'professional',
        'color_scheme': 'family_friendly'
    }
)

su.log_info("Created client:")
su.log_info(f"   - {hillcrest_client.name} ({hillcrest_client.industry})")
su.log_info(f"   - Website: {hillcrest_client.website}")
su.log_info(f"   - Location: {hillcrest_client.address}")

In [ ]:
# 4. Create Collaboration
#
# Collaboration Field Constraints (Pydantic validated):
#   collab_id:      1-50 chars, pattern: [a-zA-Z0-9_-]+
#   name:           1-100 chars
#   description:    optional, max 500 chars
#   organizations:  List[str], must be unique
#   clients:        List[str], must be unique
#   participants:   List[str], must be unique
#   status:         planning | active | on_hold | completed | cancelled (default: planning)
#   start_date:     datetime (default: now)
#   end_date:       optional, must be in the future

# Hillcrest Analytics Project - Joint project between Siege and Masai
hillcrest_collaboration = Collaboration(
    collab_id='hillcrest_analytics_2025',  # 1-50 chars, [a-zA-Z0-9_-]+
    name='Hillcrest Analytics Project 2025',  # 1-100 chars
    description='Marketing analytics project for Hillcrest Children & Family Center',  # max 500 chars
    organizations=['siege_analytics', 'masai_interactive'],  # unique list
    clients=['hillcrest_children'],        # unique list
    participants=['dheeraj_siege', 'tony_masai'],  # unique list
    status='active',                       # planning | active | on_hold | completed | cancelled
    start_date=datetime(2025, 1, 1),
    end_date=datetime(2025, 12, 31),       # must be in the future
    shared_credentials=['hillcrest_google_analytics', 'hillcrest_facebook_insights'],
    shared_databases=['hillcrest_analytics_db']
)

su.log_info("Created collaboration:")
su.log_info(f"   - {hillcrest_collaboration.name}")
su.log_info(f"   - Organizations: {', '.join(hillcrest_collaboration.organizations)}")
su.log_info(f"   - Participants: {', '.join(hillcrest_collaboration.participants)}")
su.log_info(f"   - Status: {hillcrest_collaboration.status}")

In [ ]:
# 5. Summary of Created Entities

su.log_info("All entities created successfully!")
su.log_info(f"   Organizations: {len([siege_org, masai_org, hillcrest_org])} created")
su.log_info(f"   Users: {len([dheeraj])} created")
su.log_info(f"   Collaborators: {len([tony])} created")
su.log_info(f"   Clients: {len([hillcrest_client])} created")
su.log_info(f"   Collaborations: {len([hillcrest_collaboration])} created")

su.log_info("Entity Summary:")
su.log_info(f"   - {siege_org.name} ({siege_org.org_type})")
su.log_info(f"   - {masai_org.name} ({masai_org.org_type})")
su.log_info(f"   - {hillcrest_org.name} ({hillcrest_org.org_type})")
su.log_info(f"   - {dheeraj.name} (User) - {dheeraj.role}")
su.log_info(f"   - {tony.name} (Collaborator) - {tony.collaboration_level}")
su.log_info(f"   - {hillcrest_client.name} (Client) - {hillcrest_client.industry}")
su.log_info(f"   - {hillcrest_collaboration.name} (Collaboration) - {hillcrest_collaboration.status}")

In [ ]:
# 6. Demonstrate Credential Management
#
# Credential Field Constraints (Pydantic validated):
#   name:            1-100 chars, pattern: [a-zA-Z0-9_-]+
#   credential_type: api_key | oauth_token | username_password | ssh_key | certificate | secret
#   service:         1-50 chars, pattern: [a-zA-Z0-9_-]+
#   api_key:         min 10 chars (if provided)
#
# OnePasswordCredential Field Constraints (Pydantic validated):
#   vault_id:        1-50 chars, pattern: [a-zA-Z0-9_-]+
#   item_id:         1-50 chars, pattern: [a-zA-Z0-9_-]+
#   title:           1-100 chars
#   credential_name: 1-100 chars
#   service:         1-50 chars
#   auto_sync:       bool (default: True)

# Add shared credentials for the Hillcrest project
hillcrest_ga_cred = Credential(
    name='hillcrest_google_analytics',     # 1-100 chars, [a-zA-Z0-9_-]+
    credential_type='api_key',             # api_key | oauth_token | ...
    service='google_analytics',            # 1-50 chars, [a-zA-Z0-9_-]+
    api_key='GA-987654321-HILLCREST',      # min 10 chars
    username='hillcrest@analytics.com'
)

# Add credentials to both users
dheeraj.add_credential(hillcrest_ga_cred)
tony.add_credential(hillcrest_ga_cred)

# Add 1Password credential reference
onepassword_cred = OnePasswordCredential(
    credential_name='hillcrest_facebook_token',  # 1-100 chars
    vault_id='hillcrest_vault',            # 1-50 chars, [a-zA-Z0-9_-]+
    item_id='item_123456789',              # 1-50 chars, [a-zA-Z0-9_-]+
    title='Hillcrest Facebook Token',      # 1-100 chars
    service='Facebook Business',           # 1-50 chars
    auto_sync=True                         # bool
)

dheeraj.add_onepassword_credential(onepassword_cred)

su.log_info("Credential management demonstrated:")
su.log_info(f"   Dheeraj credentials: {len(dheeraj.credentials)}")
su.log_info(f"   Tony credentials: {len(tony.credentials)}")
su.log_info(f"   Dheeraj 1Password refs: {len(dheeraj.onepassword_credentials)}")

# Show credential details
for cred in dheeraj.credentials:
    su.log_info(f"   - {cred.name} ({cred.service})")

In [ ]:
# 7. Demonstrate 1Password Methods and User-Client Relationships

# Test 1Password credential methods
su.log_info("Testing 1Password Credential Methods:")
has_ga_1p = dheeraj.has_onepassword_credential('hillcrest_facebook_token')
su.log_info(f"   Has Facebook 1Password: {has_ga_1p}")

services = dheeraj.get_onepassword_services()
su.log_info(f"   1Password services: {services}")

vaults = dheeraj.get_onepassword_vaults()
su.log_info(f"   1Password vaults: {vaults}")

# Test credential coverage analysis
coverage = dheeraj.get_credential_coverage()
su.log_info(f"   Total services covered: {coverage['summary']['total_services']}")
su.log_info(f"   Services with multiple credential types: {coverage['summary']['services_with_multiple_types']}")

# Test security recommendations
recommendations = dheeraj.get_security_recommendations()
su.log_info(f"   Security recommendations: {len(recommendations)}")
for rec in recommendations:
    su.log_info(f"     - {rec}")

su.log_info("Testing User-Client Relationships:")

# Assign Dheeraj to Hillcrest client
dheeraj.assign_client('HILL')
dheeraj.set_primary_client('HILL')

# Assign Hillcrest client to Dheeraj
hillcrest_client.assign_user('dheeraj_siege')
hillcrest_client.set_primary_user('dheeraj_siege')

su.log_info("User-Client relationships established")
su.log_info(f"   Dheeraj assigned clients: {dheeraj.get_assigned_clients()}")
su.log_info(f"   Dheeraj primary client: {dheeraj.primary_client}")
su.log_info(f"   Hillcrest assigned users: {hillcrest_client.get_assigned_users()}")
su.log_info(f"   Hillcrest primary user: {hillcrest_client.primary_user}")

su.log_info("All Person/Actor functionality demonstrated successfully!")

## Person/Actor Architecture Complete!

### **What We Built:**

#### **1. Organizations**
- **Siege Analytics**: Internal organization
- **Masai Interactive**: Partner organization  
- **Hillcrest Children & Family Center**: Client organization

#### **2. Users & Collaborators**
- **Dheeraj (User)**: Siege admin with full credentials and database access
- **Tony (Collaborator)**: Masai team member with limited access and expiration

#### **3. Client Management**
- **Hillcrest Client**: Complete with branding, preferences, and project data

#### **4. Collaboration**
- **Hillcrest Analytics Project**: Joint project between Siege and Masai
- **Shared Resources**: Credentials, databases, and access controls

#### **5. Credential Management**
- **API Keys**: Google Analytics credentials
- **OAuth Integration**: Google OAuth for Tony
- **1Password Integration**: Secure credential references
- **Database Connections**: PostgreSQL access for Dheeraj

#### **6. Import/Export**
- **Individual Profiles**: Export with/without sensitive data
- **Bulk Export**: All persons, organizations, and collaborations
- **Import**: Load profiles from YAML files

### **Key Benefits:**
- **Multi-Company Support**: Perfect for Siege + Masai collaborations
- **Secure Credential Sharing**: Controlled access to sensitive data
- **Flexible Relationships**: Users can belong to multiple organizations
- **Project Management**: Track collaborations and shared resources
- **Backup & Sharing**: Full export/import capabilities

This architecture perfectly handles your **multi-company collaborative projects** with **flexible credential sharing** and **role-based access control**!